In [29]:
#Import

In [54]:
import numpy as np
import pandas as pd
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.parameter_estimator import DiscreteMLE

In [55]:
# Read Heart Disease dataset

In [56]:
heartDisease = pd.read_csv("/home/anurag/7thSemCodes/IR/IRDataset/Assign3/heart.csv")

In [57]:
# Replace missing-value markers

In [58]:
heartDisease = heartDisease.replace("?", np.nan)

In [59]:
# Convert columns to numeric

In [60]:
for col in heartDisease.columns:
    heartDisease[col] = pd.to_numeric(
        heartDisease[col], errors="coerce"
    )


In [61]:
# Display sample instances and data types

In [62]:
print("Sample instances from the dataset:")
print(heartDisease.head())

Sample instances from the dataset:
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   63    1   3       145   233    1        0      150      0      2.3      0   
1   37    1   2       130   250    0        1      187      0      3.5      0   
2   41    0   1       130   204    0        0      172      0      1.4      2   
3   56    1   1       120   236    0        1      178      0      0.8      2   
4   57    0   0       120   354    0        1      163      1      0.6      2   

   ca  thal  target  
0   0     1       1  
1   0     2       1  
2   0     2       1  
3   0     2       1  
4   0     2       1  


In [63]:
print("\nAttributes and datatypes:")
print(heartDisease.dtypes)


Attributes and datatypes:
age           int64
sex           int64
cp            int64
trestbps      int64
chol          int64
fbs           int64
restecg       int64
thalach       int64
exang         int64
oldpeak     float64
slope         int64
ca            int64
thal          int64
target        int64
dtype: object


In [64]:
# Remove missing values

In [65]:
heartDisease = heartDisease.dropna().copy()

In [66]:
# Convert target into binary values:
# 0 = No heart disease, 1 = Heart disease

In [67]:
heartDisease["target"] = (
    heartDisease["target"] > 0
).astype(int)

In [68]:
# Discretize continuous attributes

In [69]:
continuous_columns = [
    "age", "trestbps", "chol", "thalach", "oldpeak"
]

for col in continuous_columns:
    heartDisease[col] = pd.cut(
        heartDisease[col],
        bins=3,
        labels=False,
        duplicates="drop"
    )

In [70]:
# Remove remaining missing values

In [71]:
heartDisease = heartDisease.dropna().astype(int)

In [72]:
print("\nDataset after preprocessing:")
print(heartDisease.head())
print("\nDataset shape:", heartDisease.shape)


Dataset after preprocessing:
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0    2    1   3         1     0    1        0        1      0        1      0   
1    0    1   2         1     0    0        1        2      0        1      0   
2    0    0   1         1     0    0        0        2      0        0      2   
3    1    1   1         0     0    0        1        2      0        0      2   
4    1    0   0         0     1    0        1        2      1        0      2   

   ca  thal  target  
0   0     1       1  
1   0     2       1  
2   0     2       1  
3   0     2       1  
4   0     2       1  

Dataset shape: (303, 14)


In [73]:
# Construct the Bayesian Network

In [74]:
model = DiscreteBayesianNetwork([
    ("age", "target"),
    ("sex", "target"),
    ("exang", "target"),
    ("cp", "target"),
    ("target", "restecg"),
    ("target", "chol")
])

In [75]:
# Learn Conditional Probability Distributions (CPDs)

In [77]:
print("\nLearning CPDs using Maximum Likelihood Estimation...")

model.fit(
    heartDisease,
    estimator=DiscreteMLE()
)

print("CPDs learned successfully.")


Learning CPDs using Maximum Likelihood Estimation...
CPDs learned successfully.


In [78]:
# Check model validity

In [79]:
print("\nIs the model valid?", model.check_model())


Is the model valid? True


In [80]:
# Perform inference using Variable Elimination

In [82]:
print("\nPerforming inference:")
HeartDiseasetest_infer = VariableElimination(model)
print("\nPerforming inference:")
HeartDiseasetest_infer = VariableElimination(model)

# Probability of heart disease given resting ECG evidence
q1 = HeartDiseasetest_infer.query(
    variables=["target"],
    evidence={"restecg": 1}
)

print("\n1. Probability of heart disease given restecg = 1:")
print(q1)

# Probability of heart disease given chest pain evidence
q2 = HeartDiseasetest_infer.query(
    variables=["target"],
    evidence={"cp": 2}
)

print("\n2. Probability of heart disease given cp = 2:")
print(q2)


Performing inference:

Performing inference:

1. Probability of heart disease given restecg = 1:
+-----------+---------------+
| target    |   phi(target) |
+===========+===============+
| target(0) |        0.3425 |
+-----------+---------------+
| target(1) |        0.6575 |
+-----------+---------------+

2. Probability of heart disease given cp = 2:
+-----------+---------------+
| target    |   phi(target) |
+===========+===============+
| target(0) |        0.2525 |
+-----------+---------------+
| target(1) |        0.7475 |
+-----------+---------------+
